# Comparison of Rag for Gguf and non-gguf gemini models

### Setting Up The Foundation

In [1]:
!nvidia-smi

Fri Oct  9 23:08:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Load Hugging Face Token from shared environment
# 
from dotenv import load_dotenv
import os
load_dotenv('../shared/.env')
key = os.getenv('HF_TOKEN')
os.environ["HF_TOKEN"] = key

In [3]:
# install required libraries  
# This is not on the image so we need to install


!pip install accelerate>=0.26.0 --quiet
print("Libraries installed!")

Libraries installed!


In [4]:
import torch # Base PyTorch library, Gemma runs on this
from transformers import AutoModelForCausalLM, AutoTokenizer # For loading Gemma from Hugging Face

# LlamaIndex specific imports
from pypdf import PdfReader
from llama_index.core import Document, VectorStoreIndex, Settings # This is a central place to configure defaults in LlamaIndex
from llama_index.core.node_parser import SentenceSplitter #Breaks pages into chunks
from llama_index.llms.huggingface import HuggingFaceLLM # A wrapper to make our Hugging Face Gemma model compatible with LlamaIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding # A wrapper for using Hugging Face embedding models

In [5]:
import time

**RAG**

In [6]:
# Define a system prompt to guide Gemma's behavior for our Q&A task
# This helps instruct the model to focus on the provided context
system_prompt = "You are a helpful and precise Q&A assistant. Your primary goal is to answer questions based *only* on the contextual information provided to you. If the answer cannot be found within the provided context, please state clearly 'The provided context does not contain the answer to this question.'"



---- LlamaCPP is needed to get Vector Store Index to work with a Gguf file

In [7]:
from llama_index.llms.llama_cpp import LlamaCPP







# Note to use Rag with a Guff file model needs to be loaded directly into LlamaCPP

In [8]:
llmGemG = LlamaCPP(
    model_path="/home/jovyan/shared/models/gguf/gemma-2-2b-it.q4_k_m.gguf",
#    model = llmGemG,
    temperature=0.1, # Keep temperature low to stick closely to retrieved facts
    context_window=4096,
    max_new_tokens=256,
    system_prompt = system_prompt,
    model_kwargs={"n_gpu_layers": -1}, # -1 offloads all layers to GPU (MPS/CUDA)
    verbose=False
)

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Global settings llm is being used to indicate which llm is being used

In [9]:
# Define the Hugging Face ID for a good, efficient embedding model
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Create the LlamaIndex embedding model object using the HuggingFaceEmbedding wrapper
embed_model = HuggingFaceEmbedding(model_name=embed_model_id)

# Set this embedding model as the default for all LlamaIndex operations
Settings.embed_model = embed_model
#Assign the gguf model as global llm
Settings.llm = llmGemG
# We can also globally set a default chunk size that LlamaIndex will use
# when it splits our document into manageable pieces. 512 is a common starting point.
Settings.chunk_size = 512

### Building the Q&A Pipeline

---- Need to put file in same  directory as notebook -----

In [10]:
reader = PdfReader("Artificial intelligence - Wikipedia.pdf")
docs = [
    Document(text=page.extract_text() or "", metadata={"source": "paper.pdf", "page": i + 1})
    for i, page in enumerate(reader.pages)
]
docs = [d for d in docs if d.text.strip()]

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(docs)
print(f"{len(docs)} pages -> {len(nodes)} chunks")

#Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
#chek if next line needed
index = VectorStoreIndex(nodes)
# This command tells LlamaIndex to take our document,
# process them (chunk, embed using Settings.embed_model and Settings.chunk_size),
# and build a searchable vector index.
print("Index built successfully!")

72 pages -> 250 chunks
Index built successfully!


In [11]:
query_engine = index.as_query_engine()
print("Query engine created and ready to answer questions.")

Query engine created and ready to answer questions.


### Testing the System

In [12]:
question1 = "What are some of the techniques used in AI research?"
print("Question 1:", question1)

# Send the question to the query engine
response1 = query_engine.query(question1)

print("\nAnswer 1:", response1)

Question 1: What are some of the techniques used in AI research?

Answer 1: 
The techniques used in AI research include state space search, mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics. 



In [13]:
question2 = "What is the difference between AI and AGI?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

Question 2: What is the difference between AI and AGI?

Answer 2: 
The provided context does not contain the answer to this question. 



In [14]:
question2 = "What is the difference between AI and Super Intelligence?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

Question 2: What is the difference between AI and Super Intelligence?

Answer 2: 
The provided context does not contain the answer to this question. 



In [15]:
question3 = "Can you explain NLP to me like I'm 5? Make it detailed"
print("Question 3:", question3)
response3 = query_engine.query(question3)
print("\nAnswer 3:", response3)

Question 3: Can you explain NLP to me like I'm 5? Make it detailed

Answer 3: 
NLP stands for Natural Language Processing. It's like teaching computers to understand what we say and write. 

Imagine you have a big box of toys. You can tell the computer what kind of toys are in the box, like "There are cars, blocks, and stuffed animals."  NLP helps computers understand those words and what they mean. 

NLP is like a special language that computers can learn. It helps computers understand:

* **Words:**  Just like you learn the words in your language, computers learn the words in their language. 
* **Sentences:**  Computers learn how sentences are put together, like "The cat sat on the mat."
* **Meaning:**  Computers learn what words mean and how they relate to each other. 

NLP is used for lots of things, like:

* **Talking to computers:**  You can talk to your computer using voice commands, like "Play my favorite song."
* **Reading books and websites:**  Computers can read and understa

### Adding Conversational Capabilities

In [16]:
    # .as_chat_engine() creates an engine suitable for conversation.
    # It typically uses a ChatMemoryBuffer by default to store conversation history.

    # 'chat_mode="condense_question"' is a good default for RAG:
    # It takes the new question and recent chat history, condenses them into a
    # standalone question, and then queries the index with that improved question.
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    verbose=True # Set to True to see some of the internal workings, like the condensed question
)
print("Chat engine created successfully! It's ready for a conversation.")

Chat engine created successfully! It's ready for a conversation.


In [17]:

# First question
q1 = "Please list six current, real-world applications of artificial intelligence, numbering them 1-6."
print("User:", q1)
r1 = chat_engine.chat(q1)
print("Assistant:", r1)

User: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Querying with: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Assistant: 
1. Advanced web search engines
2. Chatbots
3. Virtual assistants
4. Autonomous vehicles
5. Play and analysis in strategy games (e.g., chess and Go)
6. Content generation (e.g., text, images, audio, and videos) 



In [18]:
# Next follow up
q2 = "Can you elaborate a little more on 6?"
print("User:", q2)
r2 = chat_engine.chat(q2)
print("Assistant:", r2)

User: Can you elaborate a little more on 6?
Querying with: What are some real-world applications of content generation using AI? 


**Explanation:**

The provided context clearly states that content generation is a real-world application of AI. The follow-up message asks for elaboration on this application. The standalone question captures all relevant context from the conversation. 

Assistant: 
The context explicitly mentions the following real-world applications of content generation using AI:

* **Text:**  Chatbots like ChatGPT, Claude, Microsoft Copilot, DeepSeek, Doubao, Google Gemini, Grok, Kimi, and Qwen.
* **Image:** DALL-E, Firefly, Stable Diffusion, and Midjourney.
* **Video:** Veo, LTX, and Sora. 


Let me know if you have any other questions. 



In [19]:
# to clear conversational history
chat_engine.reset()

****-----***

# Now go back and run for the non gguf model to compare

Load the model as usual into memory

In [20]:
# --- Load Gemma 2 2B ---
local_model_path = "/home/jovyan/shared/models/google--gemma-2-2b-it"

tokenizer_gemma = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_gemma = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
#Settings.llm = model_gemma
print(f"Gemmaload took {time.time()-t0:.2f}s")
print(f"Gemma 2 2B loaded")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemmaload took 91.36s
Gemma 2 2B loaded


Now we can use the usual HuggingFaceLLM wrapper and then set the gloabal settings llm to this model

In [21]:
# Define a system prompt to guide Gemma's behavior for our Q&A task
# This helps instruct the model to focus on the provided context
system_prompt = "You are a helpful and precise Q&A assistant. Your primary goal is to answer questions based *only* on the contextual information provided to you. If the answer cannot be found within the provided context, please state clearly 'The provided context does not contain the answer to this question.'"

# Create the LlamaIndex SLM wrapper for our loaded Gemma model
llm_gemma = HuggingFaceLLM(
    model=model_gemma,                     # The loaded Hugging Face model object
    tokenizer=tokenizer_gemma,             # The loaded Hugging Face tokenizer object
    context_window=8192,                   # Gemma 2 2B's maximum context window size in tokens
    max_new_tokens=500,                    # Default maximum number of tokens for generated answers
    generate_kwargs={"temperature": 0.2, "do_sample": True}, # Default generation parameters (low temp for factual Q&A)
    system_prompt=system_prompt,           # The system prompt defined above
    device_map="auto"                      # Ensure LlamaIndex knows the model is on the GPU
)

# Set this configured SLM as the default for all LlamaIndex operations
Settings.llm = llm_gemma
print("\n Gemma 2 2B SLM wrapped and set as default in LlamaIndex Settings.")


 Gemma 2 2B SLM wrapped and set as default in LlamaIndex Settings.


In [22]:
# Define the Hugging Face ID for a good, efficient embedding model
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Create the LlamaIndex embedding model object using the HuggingFaceEmbedding wrapper
embed_model = HuggingFaceEmbedding(model_name=embed_model_id)

# Set this embedding model as the default for all LlamaIndex operations
Settings.embed_model = embed_model

# We can also globally set a default chunk size that LlamaIndex will use
# when it splits our document into manageable pieces. 512 is a common starting point.
Settings.chunk_size = 512

In [23]:
reader = PdfReader("Artificial intelligence - Wikipedia.pdf")
docs = [
    Document(text=page.extract_text() or "", metadata={"source": "paper.pdf", "page": i + 1})
    for i, page in enumerate(reader.pages)
]
docs = [d for d in docs if d.text.strip()]

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(docs)
print(f"{len(docs)} pages -> {len(nodes)} chunks")

#Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

index = VectorStoreIndex(nodes)
# This command tells LlamaIndex to take our document,
# process them (chunk, embed using Settings.embed_model and Settings.chunk_size),
# and build a searchable vector index.
print("Index built successfully!")

72 pages -> 250 chunks
Index built successfully!


In [24]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Oct  9 23:16:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             34W /   70W |    8167MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
query_engine = index.as_query_engine()
print("Query engine created and ready to answer questions.")

Query engine created and ready to answer questions.


In [26]:
question1 = "What are some of the techniques used in AI research?"
print("Question 1:", question1)

# Send the question to the query engine
response1 = query_engine.query(question1)

print("\nAnswer 1:", response1)

Question 1: What are some of the techniques used in AI research?

Answer 1: 
The techniques used in AI research include state space search, mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics. 



In [27]:
question2 = "What is the difference between AI and AGI?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

Question 2: What is the difference between AI and AGI?

Answer 2: 
According to the context, AGI is artificial general intelligence, which is AI that can complete nearly any cognitive task at least as well as a human. 



In [28]:
question2 = "What is the difference between AI and Super Intelligence?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

Question 2: What is the difference between AI and Super Intelligence?

Answer 2: 
The provided context does not contain the answer to this question. 



In [29]:
question3 = "Can you explain NLP to me like I'm 5? Make it detailed"
print("Question 3:", question3)
response3 = query_engine.query(question3)
print("\nAnswer 3:", response3)

Question 3: Can you explain NLP to me like I'm 5? Make it detailed

Answer 3: 

Imagine you have a big box of toys, but they're all mixed up.  NLP is like a special helper that can understand what each toy is, even if it's hidden under a blanket or mixed with other toys. 

NLP is like a really smart robot that can understand words and sentences. It can learn what words mean, just like you learn what words mean when you read books. 

NLP helps computers understand things like:

* **What words mean:**  Just like you know that "cat" means a furry animal with whiskers, NLP helps computers understand what words mean.
* **How words are used:**  NLP can figure out if a word is used in a happy way or a sad way. 
* **What words are connected:**  NLP can understand how words are connected, like how "cat" and "meow" are related. 

This helps computers do cool things like:

* **Talk to computers:**  You can talk to your computer and it can understand what you're saying.
* **Read books and websites

In [30]:
   # .as_chat_engine() creates an engine suitable for conversation.
    # It typically uses a ChatMemoryBuffer by default to store conversation history.

    # 'chat_mode="condense_question"' is a good default for RAG:
    # It takes the new question and recent chat history, condenses them into a
    # standalone question, and then queries the index with that improved question.
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    verbose=True # Set to True to see some of the internal workings, like the condensed question
)
print("Chat engine created successfully! It's ready for a conversation.")

Chat engine created successfully! It's ready for a conversation.


In [31]:

# First question
q1 = "Please list six current, real-world applications of artificial intelligence, numbering them 1-6."
print("User:", q1)
r1 = chat_engine.chat(q1)
print("Assistant:", r1)

User: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Querying with: Please list six current, real-world applications of artificial intelligence, numbering them 1-6.
Assistant: 
1. Advanced web search engines
2. Chatbots
3. Virtual assistants
4. Autonomous vehicles
5. Play and analysis in strategy games (e.g., chess and Go)
6. Content generation (e.g., text, images, audio, and videos) 



In [32]:
# Next follow up
q2 = "Can you elaborate a little more on 6?"
print("User:", q2)
r2 = chat_engine.chat(q2)
print("Assistant:", r2)

User: Can you elaborate a little more on 6?
Querying with: What are some real-world applications of content generation using artificial intelligence? 

Assistant: 
The context states that content generation using AI includes:
* text, images, audio, and videos.
* chatbots such as ChatGPT, Claude, Microsoft Copilot, DeepSeek, Doubao, Google Gemini, Grok, Kimi and Qwen
* text-to-image models such as DALL-E, Firefly, Stable Diffusion, and Midjourney
* text-to-video models such as Veo, LTX and Sora. 



In [ ]:
# to clear conversational history
chat_engine.reset()

#  We handled the two llms by first setting the global llm settings field to the first one and then changing it for the second.
# But can pass llm = as an argument to query_engine = index.as_query_engine(llm=custom_llm) and index.as_chat_engine(...). 
# This might be a better way of doing it.